# Experiment 05 — Formal DP-SGD Privacy–Utility Sweep + MIA

## Main goal

Compare one non-private PyTorch MLP with formally accounted DP-SGD MLPs at multiple privacy strengths while keeping the research protocol fixed.

This notebook reports:

- actual \((\varepsilon,\delta)\) and DP-SGD configuration;
- IDS utility on `KDDTest+`;
- shadow-calibrated membership-inference results;
- bootstrap uncertainty;
- group-conditioned leakage diagnostics.

## Predeclared conditions

```text
Non-private PyTorch MLP
DP-SGD target ε ≈ 8
DP-SGD target ε ≈ 4
DP-SGD target ε ≈ 2
```

`ε ≈ 1` is deliberately excluded from this first sweep. It should be added only after reviewing whether `ε ≈ 2` retains usable Recall and FNR.

## Privacy scope

The reported guarantee covers DP-SGD optimisation **conditional on the fixed preprocessing artifact**. It is not an end-to-end raw-data privacy claim.


## 1. Install and import dependencies


In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("opacus") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "opacus"])

print("Opacus is available.")


Opacus is available.


In [2]:
from __future__ import annotations

import copy
import gc
import hashlib
import json
import os
import platform
import random
import time
import warnings
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import joblib
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn

from opacus import PrivacyEngine
from opacus.validators import ModuleValidator
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

pd.set_option("display.max_columns", 120)

print({
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "opacus": __import__("opacus").__version__,
    "sklearn": sklearn.__version__,
})


{'python': '3.13.15', 'torch': '2.11.0+cpu', 'opacus': '1.6.0', 'sklearn': '1.6.1'}


## 2. Fixed protocol and run configuration


In [3]:
SEED = 42
SHADOW_SEEDS = [101, 202, 303, 404, 505]
CALIBRATION_SHADOW_SEED = 505

HIDDEN_DIMS = (64, 32)
EPOCHS = 30
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

TARGET_EPSILONS = [8.0, 4.0, 2.0]
MAX_GRAD_NORM = 1.0
ACCOUNTANT = "prv"
POISSON_SAMPLING = True
SECURE_MODE = False

BOOTSTRAP_N = 1000
MIN_GROUP_MEMBERS = 30
MIN_GROUP_NONMEMBERS = 30

RESUME = True
SAVE_INTERMEDIATE_ATTACK_DATA = True
SHADOW_PROTOCOL_VERSION = "epsilon_delta_matched_v2"

CONDITIONS = [
    {"condition": "non_private", "formal_dp": False, "target_epsilon": None},
    *[
        {
            "condition": f"dp_eps_{int(epsilon)}",
            "formal_dp": True,
            "target_epsilon": epsilon,
        }
        for epsilon in TARGET_EPSILONS
    ],
]

def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)

def stable_seed(*parts: object) -> int:
    text = "::".join(str(part) for part in parts)
    return int(hashlib.sha256(text.encode("utf-8")).hexdigest()[:8], 16)

set_all_seeds(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

display(pd.DataFrame(CONDITIONS))
print("Device:", DEVICE)


,condition,formal_dp,target_epsilon
0,non_private,False,NaN
1,dp_eps_8,True,8.0
2,dp_eps_4,True,4.0
3,dp_eps_2,True,2.0


Device: cpu


### Fair-comparison controls

Target training remains fixed. Each DP shadow now receives the same requested
`(epsilon, delta)` as its corresponding target condition. Opacus may calculate
a different shadow noise multiplier because the shadow dataset and sampling rate
are different. Every shadow's actual epsilon is recorded and checked.


## 3. Resolve Drive paths and accepted prerequisites


In [4]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

DEFAULT_PROJECT_DIR = Path("/content/drive/MyDrive/ML-DP-NID")
PROJECT_DIR = Path(os.environ.get("ML_DP_NID_DIR", DEFAULT_PROJECT_DIR))
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path.cwd()

TRAIN_FILE = PROJECT_DIR / "KDDTrain+.txt"
TEST_FILE = PROJECT_DIR / "KDDTest+.txt"

SPLIT_DIR = PROJECT_DIR / "data" / "split_indices"
TARGET_TRAIN_INDEX_FILE = SPLIT_DIR / "target_train_indices.npy"
TARGET_VALIDATION_INDEX_FILE = SPLIT_DIR / "target_validation_indices.npy"
SHADOW_POOL_INDEX_FILE = SPLIT_DIR / "shadow_pool_indices.npy"

PREPROCESSOR_FILE = PROJECT_DIR / "artifacts" / "preprocessor" / "target_mlp_preprocessor.joblib"
BASELINE_MANIFEST_FILE = PROJECT_DIR / "artifacts" / "manifests" / "baseline_mia_combined_manifest.json"
SMOKE_MANIFEST_FILE = PROJECT_DIR / "results" / "dp_sgd_smoke_test" / "dp_sgd_smoke_manifest.json"

RESULTS_DIR = PROJECT_DIR / "results" / "dp_sgd"
INTERMEDIATE_DIR = RESULTS_DIR / "intermediate"
TARGET_MODEL_DIR = PROJECT_DIR / "artifacts" / "models" / "dp_sgd_sweep"
MANIFEST_DIR = PROJECT_DIR / "artifacts" / "manifests"

for directory in [RESULTS_DIR, INTERMEDIATE_DIR, TARGET_MODEL_DIR, MANIFEST_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

required_files = [
    TRAIN_FILE,
    TEST_FILE,
    TARGET_TRAIN_INDEX_FILE,
    TARGET_VALIDATION_INDEX_FILE,
    SHADOW_POOL_INDEX_FILE,
    PREPROCESSOR_FILE,
    BASELINE_MANIFEST_FILE,
    SMOKE_MANIFEST_FILE,
]

missing_files = [str(path) for path in required_files if not path.exists()]
assert not missing_files, "Missing prerequisite files:\n" + "\n".join(missing_files)

print("Project directory:", PROJECT_DIR)
print("Result directory:", RESULTS_DIR)


Mounted at /content/drive
Project directory: /content/drive/MyDrive/ML-DP-NID
Result directory: /content/drive/MyDrive/ML-DP-NID/results/dp_sgd


In [5]:
def calculate_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def load_json(path: Path) -> dict:
    with path.open("r") as file:
        return json.load(file)


def to_json_safe(value):
    if isinstance(value, dict):
        return {str(key): to_json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_json_safe(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value


def write_strict_json(path: Path, payload: dict) -> None:
    with path.open("w") as file:
        json.dump(
            to_json_safe(payload),
            file,
            indent=2,
            allow_nan=False,
        )

baseline_manifest = load_json(BASELINE_MANIFEST_FILE)
smoke_manifest = load_json(SMOKE_MANIFEST_FILE)

assert baseline_manifest["run_mode"] == "final"
assert baseline_manifest["number_of_shadow_models"] == 5
assert smoke_manifest["parity"]["passed"] is True
assert smoke_manifest["dp"]["accountant"] == ACCOUNTANT
assert smoke_manifest["dp"]["max_grad_norm"] == MAX_GRAD_NORM

actual_train_hash = calculate_sha256(TRAIN_FILE)
actual_test_hash = calculate_sha256(TEST_FILE)
preprocessor_hash = calculate_sha256(PREPROCESSOR_FILE)

assert actual_train_hash == baseline_manifest["dataset"]["train_sha256"]
assert actual_test_hash == baseline_manifest["dataset"]["test_sha256"]

print("Accepted baseline and smoke-test prerequisites verified.")


Accepted baseline and smoke-test prerequisites verified.


## 4. Load NSL-KDD and reconstruct the locked split


In [6]:
COLUMNS = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins",
    "logged_in", "num_compromised", "root_shell", "su_attempted", "num_root",
    "num_file_creations", "num_shells", "num_access_files",
    "num_outbound_cmds", "is_host_login", "is_guest_login", "count",
    "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate",
    "srv_rerror_rate", "same_srv_rate", "diff_srv_rate",
    "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate",
    "label", "difficulty",
]

FEATURES = COLUMNS[:41]
CATEGORICAL_FEATURES = ["protocol_type", "service", "flag"]
NUMERIC_FEATURES = [column for column in FEATURES if column not in CATEGORICAL_FEATURES]

DOS_ATTACKS = {
    "back", "land", "neptune", "pod", "smurf", "teardrop",
    "apache2", "udpstorm", "processtable", "mailbomb",
}
PROBE_ATTACKS = {"satan", "ipsweep", "nmap", "portsweep", "mscan", "saint"}
R2L_ATTACKS = {
    "guess_passwd", "ftp_write", "imap", "phf", "multihop",
    "warezmaster", "warezclient", "spy", "xlock", "xsnoop",
    "snmpguess", "snmpgetattack", "httptunnel", "sendmail", "named",
}
U2R_ATTACKS = {"buffer_overflow", "loadmodule", "rootkit", "perl", "sqlattack", "xterm", "ps"}

def map_attack_family(label: str) -> str:
    label = str(label).strip().lower().rstrip(".")
    if label == "normal":
        return "Normal"
    if label in DOS_ATTACKS:
        return "DoS"
    if label in PROBE_ATTACKS:
        return "Probe"
    if label in R2L_ATTACKS:
        return "R2L"
    if label in U2R_ATTACKS:
        return "U2R"
    return "OtherAttack"

def pool_rare_families(attack_family: str) -> str:
    return attack_family if attack_family in {"Normal", "DoS", "Probe"} else "Rare"

def load_nsl_kdd(path: Path) -> pd.DataFrame:
    dataframe = pd.read_csv(path, names=COLUMNS)
    dataframe["row_id"] = np.arange(len(dataframe))
    dataframe["label_clean"] = (
        dataframe["label"].astype(str).str.strip().str.lower().str.rstrip(".")
    )
    dataframe["binary_label"] = (dataframe["label_clean"] != "normal").astype(int)
    dataframe["attack_family"] = dataframe["label_clean"].map(map_attack_family)
    dataframe["family_group"] = dataframe["attack_family"].map(pool_rare_families)
    for column in NUMERIC_FEATURES:
        dataframe[column] = pd.to_numeric(dataframe[column], errors="raise")
    return dataframe


In [7]:
train_df = load_nsl_kdd(TRAIN_FILE)
test_df = load_nsl_kdd(TEST_FILE)

target_train_indices = np.load(TARGET_TRAIN_INDEX_FILE)
target_validation_indices = np.load(TARGET_VALIDATION_INDEX_FILE)
shadow_pool_indices = np.load(SHADOW_POOL_INDEX_FILE)

assert len(target_train_indices) == 88181
assert len(target_validation_indices) == 12597
assert len(shadow_pool_indices) == 25195

all_indices = np.concatenate([
    target_train_indices,
    target_validation_indices,
    shadow_pool_indices,
])

assert len(np.unique(all_indices)) == len(train_df)
assert set(all_indices) == set(range(len(train_df)))

target_train = train_df.iloc[target_train_indices].copy().reset_index(drop=True)
target_validation = train_df.iloc[target_validation_indices].copy().reset_index(drop=True)
shadow_pool = train_df.iloc[shadow_pool_indices].copy().reset_index(drop=True)

display(pd.DataFrame([
    {
        "partition": "target_train",
        "rows": len(target_train),
        "normal": int((target_train["binary_label"] == 0).sum()),
        "attack": int((target_train["binary_label"] == 1).sum()),
    },
    {
        "partition": "target_validation",
        "rows": len(target_validation),
        "normal": int((target_validation["binary_label"] == 0).sum()),
        "attack": int((target_validation["binary_label"] == 1).sum()),
    },
    {
        "partition": "shadow_pool",
        "rows": len(shadow_pool),
        "normal": int((shadow_pool["binary_label"] == 0).sum()),
        "attack": int((shadow_pool["binary_label"] == 1).sum()),
    },
]))


,partition,rows,normal,attack
0,target_train,88181,47140,41041
1,target_validation,12597,6734,5863
2,shadow_pool,25195,13469,11726


## 5. Fixed target preprocessing and target tensors


In [8]:
def create_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def create_preprocessor() -> ColumnTransformer:
    return ColumnTransformer([
        ("numeric", MinMaxScaler(), NUMERIC_FEATURES),
        ("categorical", create_one_hot_encoder(), CATEGORICAL_FEATURES),
    ])

def to_float32_dense(values) -> np.ndarray:
    if hasattr(values, "toarray"):
        values = values.toarray()
    return np.asarray(values, dtype=np.float32)

target_preprocessor = joblib.load(PREPROCESSOR_FILE)

X_target_train = to_float32_dense(target_preprocessor.transform(target_train[FEATURES]))
X_target_validation = to_float32_dense(target_preprocessor.transform(target_validation[FEATURES]))
X_test = to_float32_dense(target_preprocessor.transform(test_df[FEATURES]))

y_target_train = target_train["binary_label"].to_numpy(dtype=np.float32).reshape(-1, 1)
y_target_validation = target_validation["binary_label"].to_numpy(dtype=np.float32).reshape(-1, 1)
y_test = test_df["binary_label"].to_numpy(dtype=np.float32).reshape(-1, 1)

TARGET_INPUT_DIM = X_target_train.shape[1]
TARGET_DELTA = 1.0 / len(X_target_train)

print({"target_input_dim": TARGET_INPUT_DIM, "target_delta": TARGET_DELTA})


{'target_input_dim': 122, 'target_delta': 1.134031140495118e-05}


## 6. Build the fixed target MIA evaluation sample


In [9]:
def sample_balanced_target_records(
    member_candidates: pd.DataFrame,
    nonmember_candidates: pd.DataFrame,
    seed: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    sampled_members = []
    sampled_nonmembers = []

    for group in ["Normal", "DoS", "Probe", "Rare"]:
        member_group = member_candidates[member_candidates["family_group"] == group]
        nonmember_group = nonmember_candidates[nonmember_candidates["family_group"] == group]
        sample_size = min(len(member_group), len(nonmember_group))
        if sample_size == 0:
            continue
        sampled_members.append(member_group.sample(sample_size, random_state=seed))
        sampled_nonmembers.append(nonmember_group.sample(sample_size, random_state=seed + 1))

    return (
        pd.concat(sampled_members, ignore_index=False),
        pd.concat(sampled_nonmembers, ignore_index=False),
    )

target_member_records, target_nonmember_records = sample_balanced_target_records(
    target_train,
    target_validation,
    SEED,
)

target_member_positions = target_member_records.index.to_numpy()
target_nonmember_positions = target_nonmember_records.index.to_numpy()

target_mia_sample_manifest = pd.concat([
    pd.DataFrame({
        "membership": 1,
        "partition": "target_train",
        "partition_position": target_member_positions,
        "row_id": target_member_records["row_id"].to_numpy(),
        "true_label": target_member_records["binary_label"].to_numpy(),
        "binary_group": np.where(
            target_member_records["binary_label"].to_numpy() == 1,
            "Attack",
            "Normal",
        ),
        "family_group": target_member_records["family_group"].to_numpy(),
    }),
    pd.DataFrame({
        "membership": 0,
        "partition": "target_validation",
        "partition_position": target_nonmember_positions,
        "row_id": target_nonmember_records["row_id"].to_numpy(),
        "true_label": target_nonmember_records["binary_label"].to_numpy(),
        "binary_group": np.where(
            target_nonmember_records["binary_label"].to_numpy() == 1,
            "Attack",
            "Normal",
        ),
        "family_group": target_nonmember_records["family_group"].to_numpy(),
    }),
], ignore_index=True)

target_mia_sample_manifest = (
    target_mia_sample_manifest.sample(frac=1, random_state=SEED).reset_index(drop=True)
)

target_mia_sample_manifest.to_csv(
    RESULTS_DIR / "target_mia_sample_manifest.csv",
    index=False,
)

display(pd.crosstab(
    target_mia_sample_manifest["family_group"],
    target_mia_sample_manifest["membership"],
))


membership,0,1
family_group,,
DoS,4593,4593
Normal,6734,6734
Probe,1166,1166
Rare,104,104


## 7. PyTorch MLP and shared training helpers


In [10]:
class BinaryMLP(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_dims: tuple[int, int] = HIDDEN_DIMS,
    ) -> None:
        super().__init__()
        hidden_1, hidden_2 = hidden_dims
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_1),
            nn.ReLU(),
            nn.Linear(hidden_1, hidden_2),
            nn.ReLU(),
            nn.Linear(hidden_2, 1),
        )

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.network(features)

def create_initial_state(input_dim: int, seed: int) -> dict:
    set_all_seeds(seed)
    return copy.deepcopy(BinaryMLP(input_dim).state_dict())

def unwrap_model(model: nn.Module) -> nn.Module:
    return getattr(model, "_module", model)

target_initial_state = create_initial_state(TARGET_INPUT_DIM, SEED)
compatibility_errors = ModuleValidator.validate(BinaryMLP(TARGET_INPUT_DIM), strict=False)
assert not compatibility_errors

print("Target MLP is Opacus-compatible.")


Target MLP is Opacus-compatible.


In [11]:
def create_dataset(features: np.ndarray, labels: np.ndarray) -> TensorDataset:
    return TensorDataset(torch.from_numpy(features), torch.from_numpy(labels))

def create_train_loader(
    features: np.ndarray,
    labels: np.ndarray,
    seed: int,
) -> DataLoader:
    generator = torch.Generator()
    generator.manual_seed(seed)
    return DataLoader(
        create_dataset(features, labels),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        generator=generator,
        drop_last=False,
    )

def create_eval_loader(features: np.ndarray, labels: np.ndarray) -> DataLoader:
    return DataLoader(
        create_dataset(features, labels),
        batch_size=1024,
        shuffle=False,
        num_workers=0,
    )

def train_one_epoch(
    model: nn.Module,
    data_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
) -> float:
    model.train()
    total_weighted_loss = 0.0
    total_examples = 0

    for features, labels in data_loader:
        features = features.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(features)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.shape[0]
        total_weighted_loss += float(loss.detach().cpu()) * batch_size
        total_examples += batch_size

    return total_weighted_loss / max(total_examples, 1)

@torch.no_grad()
def predict_probabilities(
    model: nn.Module,
    features: np.ndarray,
    labels: np.ndarray,
    device: torch.device,
) -> np.ndarray:
    model.eval()
    probability_batches = []

    for batch_features, _ in create_eval_loader(features, labels):
        batch_features = batch_features.to(device, non_blocking=True)
        probability_batches.append(torch.sigmoid(model(batch_features)).cpu().numpy())

    return np.concatenate(probability_batches, axis=0).reshape(-1)

def summarize_warnings(
    caught_warnings,
    stage: str,
    condition: str,
    shadow_seed: int | None = None,
) -> list[dict]:
    warning_counter = Counter(
        (warning.category.__name__, str(warning.message))
        for warning in caught_warnings
    )
    return [
        {
            "stage": stage,
            "condition": condition,
            "shadow_seed": shadow_seed,
            "category": category,
            "message": message,
            "occurrences": count,
        }
        for (category, message), count in warning_counter.items()
    ]


## 8. IDS threshold and metric helpers


In [12]:
def select_f2_threshold(
    y_true: np.ndarray,
    probabilities: np.ndarray,
) -> tuple[float, pd.DataFrame]:
    rows = []
    for threshold in np.arange(0.01, 1.00, 0.01):
        predictions = (probabilities >= threshold).astype(int)
        rows.append({
            "threshold": float(threshold),
            "f2": fbeta_score(y_true, predictions, beta=2, zero_division=0),
            "recall": recall_score(y_true, predictions, zero_division=0),
            "f1": f1_score(y_true, predictions, zero_division=0),
        })

    search = pd.DataFrame(rows)
    selected_threshold = float(
        search.sort_values(["f2", "recall"], ascending=False).iloc[0]["threshold"]
    )
    return selected_threshold, search

def calculate_ids_metrics(
    condition: str,
    formal_dp: bool,
    actual_epsilon: float | None,
    split_name: str,
    threshold_policy: str,
    y_true: np.ndarray,
    probabilities: np.ndarray,
    threshold: float,
) -> dict:
    predictions = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()

    return {
        "condition": condition,
        "formal_dp": formal_dp,
        "actual_epsilon": actual_epsilon,
        "delta": TARGET_DELTA if formal_dp else np.nan,
        "split": split_name,
        "threshold_policy": threshold_policy,
        "threshold": float(threshold),
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "f1": f1_score(y_true, predictions, zero_division=0),
        "fnr": fn / (fn + tp) if fn + tp else np.nan,
        "fpr": fp / (fp + tn) if fp + tn else np.nan,
        "roc_auc": roc_auc_score(y_true, probabilities),
        "pr_auc": average_precision_score(y_true, probabilities),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


## 9. Train or resume the four target conditions


In [13]:
@dataclass
class TargetConditionResult:
    condition: str
    formal_dp: bool
    target_epsilon: float | None
    actual_epsilon: float | None
    delta: float | None
    noise_multiplier: float | None
    max_grad_norm: float | None
    sample_rate: float | None
    model_state_path: Path
    ids_results: pd.DataFrame
    target_mia_features: pd.DataFrame
    training_history: pd.DataFrame
    warning_rows: list[dict]

def condition_paths(condition: str) -> dict[str, Path]:
    condition_dir = INTERMEDIATE_DIR / condition
    condition_dir.mkdir(parents=True, exist_ok=True)
    return {
        "dir": condition_dir,
        "state": TARGET_MODEL_DIR / f"{condition}.pt",
        "config": condition_dir / "target_config.json",
        "ids": condition_dir / "target_ids_results.csv",
        "mia": condition_dir / "target_mia_features.csv",
        "history": condition_dir / "target_training_history.csv",
        "threshold": condition_dir / "target_threshold_search.csv",
        "warnings": condition_dir / "target_warnings.csv",
    }

def build_target_mia_features(
    condition: str,
    train_probabilities: np.ndarray,
    validation_probabilities: np.ndarray,
) -> pd.DataFrame:
    probability_lookup = pd.concat([
        pd.DataFrame({
            "membership": 1,
            "partition_position": target_member_positions,
            "prob_attack": train_probabilities[target_member_positions],
        }),
        pd.DataFrame({
            "membership": 0,
            "partition_position": target_nonmember_positions,
            "prob_attack": validation_probabilities[target_nonmember_positions],
        }),
    ], ignore_index=True)

    merged = target_mia_sample_manifest.merge(
        probability_lookup,
        on=["membership", "partition_position"],
        how="left",
        validate="one_to_one",
    )

    true_labels = merged["true_label"].to_numpy(dtype=int)
    prob_attack = merged["prob_attack"].to_numpy()
    true_class_probability = np.where(true_labels == 1, prob_attack, 1 - prob_attack)
    true_class_probability = np.clip(true_class_probability, 1e-12, 1 - 1e-12)

    merged.insert(0, "condition", condition)
    merged["confidence"] = np.maximum(prob_attack, 1 - prob_attack)
    merged["loss"] = -np.log(true_class_probability)
    merged["correctness"] = (
        (prob_attack >= 0.5).astype(int) == true_labels
    ).astype(int)

    return merged


In [14]:
def train_target_condition(condition_spec: dict) -> TargetConditionResult:
    condition = condition_spec["condition"]
    formal_dp = condition_spec["formal_dp"]
    target_epsilon = condition_spec["target_epsilon"]
    paths = condition_paths(condition)

    resume_files = [paths["state"], paths["config"], paths["ids"], paths["mia"], paths["history"]]

    if RESUME and all(path.exists() for path in resume_files):
        saved_config = load_json(paths["config"])
        warning_rows = (
            pd.read_csv(paths["warnings"]).to_dict(orient="records")
            if paths["warnings"].exists()
            else []
        )
        print(f"Resuming completed target condition: {condition}")
        return TargetConditionResult(
            condition=condition,
            formal_dp=formal_dp,
            target_epsilon=target_epsilon,
            actual_epsilon=saved_config.get("actual_epsilon"),
            delta=saved_config.get("delta"),
            noise_multiplier=saved_config.get("noise_multiplier"),
            max_grad_norm=saved_config.get("max_grad_norm"),
            sample_rate=saved_config.get("sample_rate"),
            model_state_path=paths["state"],
            ids_results=pd.read_csv(paths["ids"]),
            target_mia_features=pd.read_csv(paths["mia"]),
            training_history=pd.read_csv(paths["history"]),
            warning_rows=warning_rows,
        )

    set_all_seeds(SEED)
    model = BinaryMLP(TARGET_INPUT_DIM)
    model.load_state_dict(target_initial_state)
    model = model.to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    criterion = nn.BCEWithLogitsLoss()
    train_loader = create_train_loader(X_target_train, y_target_train, SEED)

    privacy_engine = None
    actual_epsilon = None
    noise_multiplier = None
    actual_max_grad_norm = None
    sample_rate = None
    warning_rows = []

    if formal_dp:
        privacy_engine = PrivacyEngine(accountant=ACCOUNTANT, secure_mode=SECURE_MODE)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("once")
            model, optimizer, train_loader = privacy_engine.make_private_with_epsilon(
                module=model,
                optimizer=optimizer,
                criterion=criterion,
                data_loader=train_loader,
                target_epsilon=target_epsilon,
                target_delta=TARGET_DELTA,
                epochs=EPOCHS,
                max_grad_norm=MAX_GRAD_NORM,
                poisson_sampling=POISSON_SAMPLING,
                clipping="flat",
                loss_reduction="mean",
            )
        warning_rows.extend(
            summarize_warnings(caught, "target_make_private", condition)
        )
        noise_multiplier = float(optimizer.noise_multiplier)
        actual_max_grad_norm = float(optimizer.max_grad_norm)
        sample_rate = float(getattr(train_loader, "sample_rate", 1.0 / len(train_loader)))

    history_rows = []
    started_at = time.time()

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("once")
        for epoch in range(1, EPOCHS + 1):
            epoch_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
            epsilon_after_epoch = (
                float(privacy_engine.get_epsilon(TARGET_DELTA))
                if formal_dp
                else np.nan
            )
            history_rows.append({
                "condition": condition,
                "epoch": epoch,
                "train_loss": epoch_loss,
                "epsilon": epsilon_after_epoch,
                "delta": TARGET_DELTA if formal_dp else np.nan,
            })

            if epoch == 1 or epoch % 5 == 0 or epoch == EPOCHS:
                epsilon_text = f", epsilon={epsilon_after_epoch:.4f}" if formal_dp else ""
                print(
                    f"{condition}: epoch {epoch:02d}/{EPOCHS}, "
                    f"loss={epoch_loss:.6f}{epsilon_text}"
                )

    warning_rows.extend(summarize_warnings(caught, "target_training", condition))
    training_seconds = time.time() - started_at

    if formal_dp:
        actual_epsilon = float(privacy_engine.get_epsilon(TARGET_DELTA))
        assert np.isfinite(actual_epsilon)
        assert 0 < actual_epsilon <= target_epsilon + 0.10

    validation_probabilities = predict_probabilities(
        model,
        X_target_validation,
        y_target_validation,
        DEVICE,
    )
    test_probabilities = predict_probabilities(model, X_test, y_test, DEVICE)
    train_probabilities = predict_probabilities(
        model,
        X_target_train,
        y_target_train,
        DEVICE,
    )

    selected_threshold, threshold_search = select_f2_threshold(
        y_target_validation.reshape(-1).astype(int),
        validation_probabilities,
    )

    ids_results = pd.DataFrame([
        calculate_ids_metrics(
            condition,
            formal_dp,
            actual_epsilon,
            "target_validation",
            "default_0_5",
            y_target_validation.reshape(-1).astype(int),
            validation_probabilities,
            0.5,
        ),
        calculate_ids_metrics(
            condition,
            formal_dp,
            actual_epsilon,
            "KDDTest+",
            "default_0_5",
            y_test.reshape(-1).astype(int),
            test_probabilities,
            0.5,
        ),
        calculate_ids_metrics(
            condition,
            formal_dp,
            actual_epsilon,
            "KDDTest+",
            "validation_selected_F2",
            y_test.reshape(-1).astype(int),
            test_probabilities,
            selected_threshold,
        ),
    ])

    target_mia_features = build_target_mia_features(
        condition,
        train_probabilities,
        validation_probabilities,
    )
    training_history = pd.DataFrame(history_rows)

    torch.save(unwrap_model(model).state_dict(), paths["state"])
    ids_results.to_csv(paths["ids"], index=False)
    target_mia_features.to_csv(paths["mia"], index=False)
    training_history.to_csv(paths["history"], index=False)
    threshold_search.to_csv(paths["threshold"], index=False)

    pd.DataFrame(
        warning_rows,
        columns=[
            "stage",
            "condition",
            "shadow_seed",
            "category",
            "message",
            "occurrences",
        ],
    ).to_csv(paths["warnings"], index=False)

    target_config = {
        "condition": condition,
        "formal_dp": formal_dp,
        "target_epsilon": target_epsilon,
        "actual_epsilon": actual_epsilon,
        "delta": TARGET_DELTA if formal_dp else None,
        "noise_multiplier": noise_multiplier,
        "max_grad_norm": actual_max_grad_norm,
        "sample_rate": sample_rate,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "optimizer": "Adam",
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "accountant": ACCOUNTANT if formal_dp else None,
        "training_seconds": training_seconds,
        "selected_threshold": selected_threshold,
        "model_state_path": str(paths["state"]),
    }
    write_strict_json(paths["config"], target_config)

    del model, optimizer, train_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return TargetConditionResult(
        condition=condition,
        formal_dp=formal_dp,
        target_epsilon=target_epsilon,
        actual_epsilon=actual_epsilon,
        delta=TARGET_DELTA if formal_dp else None,
        noise_multiplier=noise_multiplier,
        max_grad_norm=actual_max_grad_norm,
        sample_rate=sample_rate,
        model_state_path=paths["state"],
        ids_results=ids_results,
        target_mia_features=target_mia_features,
        training_history=training_history,
        warning_rows=warning_rows,
    )


In [15]:
target_condition_results = {}

for condition_spec in CONDITIONS:
    result = train_target_condition(condition_spec)
    target_condition_results[result.condition] = result

dp_sgd_ids_results = pd.concat(
    [result.ids_results for result in target_condition_results.values()],
    ignore_index=True,
)
dp_sgd_ids_results.to_csv(
    RESULTS_DIR / "dp_sgd_ids_results.csv",
    index=False,
)

display(
    dp_sgd_ids_results[
        (dp_sgd_ids_results["split"] == "KDDTest+")
        & (dp_sgd_ids_results["threshold_policy"] == "validation_selected_F2")
    ][[
        "condition",
        "actual_epsilon",
        "threshold",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "fnr",
        "fpr",
        "roc_auc",
        "pr_auc",
    ]]
)


Resuming completed target condition: non_private
Resuming completed target condition: dp_eps_8
Resuming completed target condition: dp_eps_4
Resuming completed target condition: dp_eps_2


,condition,actual_epsilon,threshold,accuracy,precision,recall,f1,fnr,fpr,roc_auc,pr_auc
2,non_private,NaN,0.29,0.816537,0.958945,0.708018,0.814596,0.291982,0.040058,0.910364,0.935714
5,dp_eps_8,7.993649,0.03,0.801278,0.920129,0.712772,0.803284,0.287228,0.081763,0.837541,0.896710
8,dp_eps_4,3.998267,0.02,0.806645,0.916290,0.726720,0.810569,0.273280,0.087736,0.838832,0.897761
11,dp_eps_2,1.999038,0.03,0.785353,0.916875,0.685031,0.784176,0.314969,0.082072,0.833445,0.895967


## 10. Create condition-level target configuration table


In [16]:
target_config_rows = []

for condition, result in target_condition_results.items():
    config = load_json(condition_paths(condition)["config"])
    target_config_rows.append({
        "condition": condition,
        "formal_dp": result.formal_dp,
        "target_epsilon": result.target_epsilon,
        "actual_epsilon": result.actual_epsilon,
        "delta": result.delta,
        "noise_multiplier": result.noise_multiplier,
        "max_grad_norm": result.max_grad_norm,
        "batch_size": BATCH_SIZE,
        "sample_rate": result.sample_rate,
        "epochs": EPOCHS,
        "optimizer": "Adam",
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "accountant": ACCOUNTANT if result.formal_dp else None,
        "poisson_sampling": POISSON_SAMPLING if result.formal_dp else False,
        "secure_mode": SECURE_MODE if result.formal_dp else False,
        "training_seconds": config["training_seconds"],
        "selected_threshold": config["selected_threshold"],
        "model_state_path": str(result.model_state_path),
        "model_sha256": calculate_sha256(result.model_state_path),
    })

dp_sgd_configs = pd.DataFrame(target_config_rows)
dp_sgd_configs.to_csv(RESULTS_DIR / "dp_sgd_configs.csv", index=False)
display(dp_sgd_configs)


,condition,formal_dp,target_epsilon,actual_epsilon,delta,noise_multiplier,max_grad_norm,batch_size,sample_rate,epochs,optimizer,learning_rate,weight_decay,accountant,poisson_sampling,secure_mode,training_seconds,selected_threshold,model_state_path,model_sha256
0,non_private,False,NaN,NaN,NaN,NaN,NaN,256,NaN,30,Adam,0.001,0.0001,None,False,False,41.014075,0.29,/content/drive/MyDrive/ML-DP-NID/artifacts/mod...,c380553509ec2b66f3d8734a12905688e9a9d7178d4207...
1,dp_eps_8,True,8.0,7.993649,0.000011,0.560303,1.0,256,0.002899,30,Adam,0.001,0.0001,prv,True,False,91.873783,0.03,/content/drive/MyDrive/ML-DP-NID/artifacts/mod...,affc8ebbbed875a98295e7c169c33211961f46e4014057...
2,dp_eps_4,True,4.0,3.998267,0.000011,0.681763,1.0,256,0.002899,30,Adam,0.001,0.0001,prv,True,False,85.546901,0.02,/content/drive/MyDrive/ML-DP-NID/artifacts/mod...,7ca77b11bb5dc47f6f0549b6ef77019c2061916bd3be77...
3,dp_eps_2,True,2.0,1.999038,0.000011,0.881348,1.0,256,0.002899,30,Adam,0.001,0.0001,prv,True,False,85.470713,0.03,/content/drive/MyDrive/ML-DP-NID/artifacts/mod...,1757fc45b5e89df8d8d46f84a981ec1319d949018c5542...


## 11. Shadow-model split and feature extraction helpers


In [17]:
def create_shadow_split(
    dataframe: pd.DataFrame,
    seed: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    stratification_key = (
        dataframe["binary_label"].astype(str)
        + "__"
        + dataframe["attack_family"]
    )
    member_indices, nonmember_indices = train_test_split(
        np.arange(len(dataframe)),
        test_size=0.50,
        random_state=seed,
        stratify=stratification_key,
    )
    return (
        dataframe.iloc[member_indices].copy(),
        dataframe.iloc[nonmember_indices].copy(),
    )

def create_mia_feature_frame(
    condition: str,
    source: str,
    shadow_seed: int,
    membership: int,
    dataframe: pd.DataFrame,
    probabilities: np.ndarray,
) -> pd.DataFrame:
    true_labels = dataframe["binary_label"].to_numpy(dtype=int)
    true_class_probability = np.where(
        true_labels == 1,
        probabilities,
        1 - probabilities,
    )
    true_class_probability = np.clip(
        true_class_probability,
        1e-12,
        1 - 1e-12,
    )

    return pd.DataFrame({
        "condition": condition,
        "source": source,
        "shadow_seed": shadow_seed,
        "row_id": dataframe["row_id"].to_numpy(),
        "membership": membership,
        "true_label": true_labels,
        "binary_group": np.where(true_labels == 1, "Attack", "Normal"),
        "family_group": dataframe["family_group"].to_numpy(),
        "prob_attack": probabilities,
        "confidence": np.maximum(probabilities, 1 - probabilities),
        "loss": -np.log(true_class_probability),
        "correctness": (
            (probabilities >= 0.5).astype(int) == true_labels
        ).astype(int),
    })


## 12. Train or resume condition-matched shadow models


In [18]:
def shadow_cache_directory(condition: str) -> Path:
    cache_name = (
        "shadow_outputs"
        if condition == "non_private"
        else f"shadow_outputs_{SHADOW_PROTOCOL_VERSION}"
    )
    shadow_dir = INTERMEDIATE_DIR / condition / cache_name
    shadow_dir.mkdir(parents=True, exist_ok=True)
    return shadow_dir


def shadow_output_path(condition: str, shadow_seed: int) -> Path:
    return shadow_cache_directory(condition) / f"shadow_{shadow_seed}_mia_features.csv"

def shadow_config_path(condition: str, shadow_seed: int) -> Path:
    return shadow_cache_directory(condition) / f"shadow_{shadow_seed}_config.json"

def train_shadow_condition(
    target_result: TargetConditionResult,
    shadow_seed: int,
) -> tuple[pd.DataFrame, dict, list[dict]]:
    condition = target_result.condition
    output_path = shadow_output_path(condition, shadow_seed)
    config_path = shadow_config_path(condition, shadow_seed)

    if RESUME and output_path.exists() and config_path.exists():
        saved_config = load_json(config_path)
        current = (
            not target_result.formal_dp
            or (
                saved_config.get("shadow_protocol_version")
                == SHADOW_PROTOCOL_VERSION
                and np.isclose(
                    float(saved_config["requested_epsilon"]),
                    float(target_result.target_epsilon),
                )
                and np.isclose(
                    float(saved_config["requested_delta"]),
                    TARGET_DELTA,
                )
            )
        )
        if current:
            print(f"Resuming {condition}, shadow {shadow_seed}")
            return pd.read_csv(output_path), saved_config, []
        print(f"Ignoring stale cache for {condition}, shadow {shadow_seed}")

    shadow_members, shadow_nonmembers = create_shadow_split(shadow_pool, shadow_seed)
    shadow_preprocessor = create_preprocessor()

    X_shadow_members = to_float32_dense(
        shadow_preprocessor.fit_transform(shadow_members[FEATURES])
    )
    X_shadow_nonmembers = to_float32_dense(
        shadow_preprocessor.transform(shadow_nonmembers[FEATURES])
    )

    y_shadow_members = (
        shadow_members["binary_label"].to_numpy(dtype=np.float32).reshape(-1, 1)
    )
    y_shadow_nonmembers = (
        shadow_nonmembers["binary_label"].to_numpy(dtype=np.float32).reshape(-1, 1)
    )

    input_dim = X_shadow_members.shape[1]
    model = BinaryMLP(input_dim)
    model.load_state_dict(create_initial_state(input_dim, shadow_seed))
    model = model.to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    criterion = nn.BCEWithLogitsLoss()
    train_loader = create_train_loader(
        X_shadow_members,
        y_shadow_members,
        shadow_seed,
    )

    privacy_engine = None
    requested_epsilon = None
    requested_delta = None
    shadow_actual_epsilon = None
    shadow_noise_multiplier = None
    shadow_sample_rate = None
    warning_rows = []

    if target_result.formal_dp:
        requested_epsilon = float(target_result.target_epsilon)
        requested_delta = float(TARGET_DELTA)
        privacy_engine = PrivacyEngine(accountant=ACCOUNTANT, secure_mode=SECURE_MODE)

        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("once")
            model, optimizer, train_loader = privacy_engine.make_private_with_epsilon(
                module=model,
                optimizer=optimizer,
                criterion=criterion,
                data_loader=train_loader,
                target_epsilon=requested_epsilon,
                target_delta=requested_delta,
                epochs=EPOCHS,
                max_grad_norm=MAX_GRAD_NORM,
                poisson_sampling=POISSON_SAMPLING,
                clipping="flat",
                loss_reduction="mean",
            )

        warning_rows.extend(
            summarize_warnings(
                caught,
                "shadow_make_private",
                condition,
                shadow_seed,
            )
        )
        shadow_noise_multiplier = float(optimizer.noise_multiplier)
        shadow_sample_rate = float(
            getattr(train_loader, "sample_rate", 1.0 / len(train_loader))
        )

    history_rows = []
    started_at = time.time()

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("once")
        for epoch in range(1, EPOCHS + 1):
            epoch_loss = train_one_epoch(
                model,
                train_loader,
                optimizer,
                criterion,
                DEVICE,
            )
            shadow_epsilon_after_epoch = (
                float(privacy_engine.get_epsilon(requested_delta))
                if target_result.formal_dp
                else np.nan
            )
            history_rows.append({
                "condition": condition,
                "shadow_seed": shadow_seed,
                "epoch": epoch,
                "train_loss": epoch_loss,
                "shadow_epsilon": shadow_epsilon_after_epoch,
                "shadow_delta": requested_delta,
            })

            if epoch == 1 or epoch % 10 == 0 or epoch == EPOCHS:
                print(
                    f"{condition}, shadow {shadow_seed}: "
                    f"epoch {epoch:02d}/{EPOCHS}, loss={epoch_loss:.6f}"
                )

    warning_rows.extend(
        summarize_warnings(
            caught,
            "shadow_training",
            condition,
            shadow_seed,
        )
    )
    training_seconds = time.time() - started_at

    if target_result.formal_dp:
        shadow_actual_epsilon = float(
            privacy_engine.get_epsilon(requested_delta)
        )
        assert abs(shadow_actual_epsilon - requested_epsilon) <= 0.10

    member_probabilities = predict_probabilities(
        model,
        X_shadow_members,
        y_shadow_members,
        DEVICE,
    )
    nonmember_probabilities = predict_probabilities(
        model,
        X_shadow_nonmembers,
        y_shadow_nonmembers,
        DEVICE,
    )

    shadow_features = pd.concat([
        create_mia_feature_frame(
            condition,
            "shadow_member",
            shadow_seed,
            1,
            shadow_members,
            member_probabilities,
        ),
        create_mia_feature_frame(
            condition,
            "shadow_nonmember",
            shadow_seed,
            0,
            shadow_nonmembers,
            nonmember_probabilities,
        ),
    ], ignore_index=True)

    shadow_features = (
        shadow_features.sample(frac=1, random_state=shadow_seed).reset_index(drop=True)
    )
    shadow_features.to_csv(output_path, index=False)
    pd.DataFrame(history_rows).to_csv(
        output_path.with_name(f"shadow_{shadow_seed}_training_history.csv"),
        index=False,
    )

    shadow_config = {
        "condition": condition,
        "shadow_seed": shadow_seed,
        "formal_dp": target_result.formal_dp,
        "member_rows": len(shadow_members),
        "nonmember_rows": len(shadow_nonmembers),
        "input_dim": input_dim,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "shadow_protocol_version": (
            SHADOW_PROTOCOL_VERSION
            if target_result.formal_dp
            else "accepted_non_private_v1"
        ),
        "requested_epsilon": requested_epsilon,
        "requested_delta": requested_delta,
        "target_actual_epsilon": target_result.actual_epsilon,
        "target_noise_multiplier": target_result.noise_multiplier,
        "shadow_noise_multiplier": shadow_noise_multiplier,
        "absolute_epsilon_error": (
            abs(shadow_actual_epsilon - requested_epsilon)
            if target_result.formal_dp
            else None
        ),
        "max_grad_norm": MAX_GRAD_NORM if target_result.formal_dp else None,
        "sample_rate": shadow_sample_rate,
        "shadow_actual_epsilon": shadow_actual_epsilon,
        "shadow_delta": requested_delta,
        "training_seconds": training_seconds,
    }

    write_strict_json(config_path, shadow_config)

    del model, optimizer, train_loader, shadow_preprocessor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return shadow_features, shadow_config, warning_rows


In [19]:
shadow_data_by_condition = {}
shadow_config_rows = []
all_warning_rows = []

for condition, target_result in target_condition_results.items():
    condition_frames = []

    for shadow_seed in SHADOW_SEEDS:
        shadow_features, shadow_config, warning_rows = train_shadow_condition(
            target_result,
            shadow_seed,
        )
        condition_frames.append(shadow_features)
        shadow_config_rows.append(shadow_config)
        all_warning_rows.extend(warning_rows)

    shadow_data_by_condition[condition] = pd.concat(
        condition_frames,
        ignore_index=True,
    )

shadow_configs = pd.DataFrame(shadow_config_rows)
shadow_configs.to_csv(RESULTS_DIR / "shadow_model_configs.csv", index=False)

print("All target-condition shadow protocols completed.")


dp_shadow_configs = shadow_configs[
    shadow_configs["formal_dp"] == True
].copy()

shadow_budget_gate = bool(
    (dp_shadow_configs["absolute_epsilon_error"].astype(float) <= 0.10).all()
    and np.allclose(
        dp_shadow_configs["shadow_delta"].astype(float),
        TARGET_DELTA,
    )
)

display(dp_shadow_configs[[
    "condition",
    "shadow_seed",
    "requested_epsilon",
    "shadow_actual_epsilon",
    "absolute_epsilon_error",
    "requested_delta",
    "target_noise_multiplier",
    "shadow_noise_multiplier",
]])

assert shadow_budget_gate
print("DP shadow epsilon/delta matching gate:", shadow_budget_gate)


Resuming non_private, shadow 101
Resuming non_private, shadow 202
Resuming non_private, shadow 303
Resuming non_private, shadow 404
Resuming non_private, shadow 505
Resuming dp_eps_8, shadow 101
Resuming dp_eps_8, shadow 202
Resuming dp_eps_8, shadow 303
Resuming dp_eps_8, shadow 404
Resuming dp_eps_8, shadow 505
Resuming dp_eps_4, shadow 101
Resuming dp_eps_4, shadow 202
Resuming dp_eps_4, shadow 303
Resuming dp_eps_4, shadow 404
Resuming dp_eps_4, shadow 505
Resuming dp_eps_2, shadow 101
Resuming dp_eps_2, shadow 202
Resuming dp_eps_2, shadow 303
Resuming dp_eps_2, shadow 404
Resuming dp_eps_2, shadow 505
All target-condition shadow protocols completed.


,condition,shadow_seed,requested_epsilon,shadow_actual_epsilon,absolute_epsilon_error,requested_delta,target_noise_multiplier,shadow_noise_multiplier
5,dp_eps_8,101,8.0,7.995812,0.004188,0.000011,0.560303,0.792542
6,dp_eps_8,202,8.0,7.995812,0.004188,0.000011,0.560303,0.792542
7,dp_eps_8,303,8.0,7.995812,0.004188,0.000011,0.560303,0.792542
8,dp_eps_8,404,8.0,7.995812,0.004188,0.000011,0.560303,0.792542
9,dp_eps_8,505,8.0,7.995812,0.004188,0.000011,0.560303,0.792542
10,dp_eps_4,101,4.0,3.995640,0.004360,0.000011,0.681763,1.101074
11,dp_eps_4,202,4.0,3.995640,0.004360,0.000011,0.681763,1.101074
12,dp_eps_4,303,4.0,3.995640,0.004360,0.000011,0.681763,1.101074
13,dp_eps_4,404,4.0,3.995640,0.004360,0.000011,0.681763,1.101074
14,dp_eps_4,505,4.0,3.995640,0.004360,0.000011,0.681763,1.101074


DP shadow epsilon/delta matching gate: True


## 13. MIA metric and bootstrap helpers


In [20]:
def calculate_mia_advantage(
    membership_labels: np.ndarray,
    membership_scores: np.ndarray,
) -> float:
    false_positive_rates, true_positive_rates, _ = roc_curve(
        membership_labels,
        membership_scores,
    )
    return float(np.max(true_positive_rates - false_positive_rates))

def calculate_tpr_at_fpr(
    membership_labels: np.ndarray,
    membership_scores: np.ndarray,
    maximum_fpr: float,
) -> float:
    false_positive_rates, true_positive_rates, _ = roc_curve(
        membership_labels,
        membership_scores,
    )
    eligible = false_positive_rates <= maximum_fpr
    return float(np.max(true_positive_rates[eligible])) if np.any(eligible) else 0.0

def find_balanced_accuracy_threshold(
    membership_labels: np.ndarray,
    membership_scores: np.ndarray,
) -> tuple[float, float]:
    candidate_thresholds = np.unique(
        np.quantile(membership_scores, np.linspace(0.001, 0.999, 400))
    )

    best_threshold = float(candidate_thresholds[0])
    best_balanced_accuracy = -1.0

    for threshold in candidate_thresholds:
        predictions = membership_scores >= threshold
        balanced_accuracy = balanced_accuracy_score(
            membership_labels,
            predictions,
        )
        if balanced_accuracy > best_balanced_accuracy:
            best_threshold = float(threshold)
            best_balanced_accuracy = float(balanced_accuracy)

    return best_threshold, best_balanced_accuracy

def bootstrap_confidence_interval(
    membership_labels: np.ndarray,
    membership_scores: np.ndarray,
    metric_function: Callable,
    seed: int,
    bootstrap_samples: int = BOOTSTRAP_N,
) -> tuple[float, float]:
    random_generator = np.random.default_rng(seed)
    bootstrap_values = []

    for _ in range(bootstrap_samples):
        sample_indices = random_generator.integers(
            0,
            len(membership_labels),
            len(membership_labels),
        )
        sampled_labels = membership_labels[sample_indices]
        sampled_scores = membership_scores[sample_indices]

        if len(np.unique(sampled_labels)) < 2:
            continue

        bootstrap_values.append(
            metric_function(sampled_labels, sampled_scores)
        )

    if not bootstrap_values:
        return np.nan, np.nan

    lower, upper = np.quantile(bootstrap_values, [0.025, 0.975])
    return float(lower), float(upper)



def paired_bootstrap_metric_difference(
    labels: np.ndarray,
    reference_scores: np.ndarray,
    comparison_scores: np.ndarray,
    metric_function: Callable,
    seed: int,
) -> tuple[float, float, float]:
    rng = np.random.default_rng(seed)
    estimate = float(
        metric_function(labels, comparison_scores)
        - metric_function(labels, reference_scores)
    )
    values = []
    for _ in range(BOOTSTRAP_N):
        indices = rng.integers(0, len(labels), len(labels))
        sampled_labels = labels[indices]
        if len(np.unique(sampled_labels)) < 2:
            continue
        values.append(
            metric_function(sampled_labels, comparison_scores[indices])
            - metric_function(sampled_labels, reference_scores[indices])
        )
    low, high = np.quantile(values, [0.025, 0.975])
    return estimate, float(low), float(high)


## 14. Calibrate condition-matched MIA attackers


In [21]:
@dataclass
class CalibratedAttack:
    condition: str
    threat_model: str
    attack_model: str
    feature_set: str
    operating_threshold: float
    shadow_calibration_auc: float
    score_function: Callable[[pd.DataFrame], np.ndarray]

def build_threshold_attack(
    condition: str,
    attack_calibration: pd.DataFrame,
    threat_model: str,
    attack_model: str,
    feature_column: str,
    score_direction: float,
) -> CalibratedAttack:
    calibration_labels = attack_calibration["membership"].to_numpy()
    calibration_scores = (
        score_direction * attack_calibration[feature_column].to_numpy()
    )
    operating_threshold, _ = find_balanced_accuracy_threshold(
        calibration_labels,
        calibration_scores,
    )

    return CalibratedAttack(
        condition=condition,
        threat_model=threat_model,
        attack_model=attack_model,
        feature_set=feature_column,
        operating_threshold=operating_threshold,
        shadow_calibration_auc=roc_auc_score(
            calibration_labels,
            calibration_scores,
        ),
        score_function=lambda dataframe: (
            score_direction * dataframe[feature_column].to_numpy()
        ),
    )

def build_learned_attack(
    condition: str,
    attack_train: pd.DataFrame,
    attack_calibration: pd.DataFrame,
    threat_model: str,
    attack_model: str,
    feature_columns: list[str],
) -> CalibratedAttack:
    if attack_model == "logistic_regression":
        model = Pipeline([
            ("scale", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    max_iter=1000,
                    class_weight="balanced",
                    random_state=SEED,
                ),
            ),
        ])
    elif attack_model == "random_forest":
        model = RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=5,
            class_weight="balanced_subsample",
            random_state=SEED,
            n_jobs=-1,
        )
    else:
        raise ValueError(f"Unsupported attacker: {attack_model}")

    model.fit(
        attack_train[feature_columns],
        attack_train["membership"],
    )
    calibration_scores = model.predict_proba(
        attack_calibration[feature_columns]
    )[:, 1]
    operating_threshold, _ = find_balanced_accuracy_threshold(
        attack_calibration["membership"].to_numpy(),
        calibration_scores,
    )

    return CalibratedAttack(
        condition=condition,
        threat_model=threat_model,
        attack_model=attack_model,
        feature_set=" + ".join(feature_columns),
        operating_threshold=operating_threshold,
        shadow_calibration_auc=roc_auc_score(
            attack_calibration["membership"],
            calibration_scores,
        ),
        score_function=lambda dataframe: (
            model.predict_proba(dataframe[feature_columns])[:, 1]
        ),
    )


In [22]:
def calibrate_condition_attacks(
    condition: str,
    shadow_data: pd.DataFrame,
) -> tuple[list[CalibratedAttack], pd.DataFrame]:
    attack_train = shadow_data[
        shadow_data["shadow_seed"] != CALIBRATION_SHADOW_SEED
    ].copy()
    attack_calibration = shadow_data[
        shadow_data["shadow_seed"] == CALIBRATION_SHADOW_SEED
    ].copy()

    assert set(attack_train["shadow_seed"].unique()).isdisjoint(
        set(attack_calibration["shadow_seed"].unique())
    )

    attacks = [
        build_threshold_attack(
            condition,
            attack_calibration,
            "score_only_black_box",
            "confidence_threshold",
            "confidence",
            1.0,
        ),
        build_learned_attack(
            condition,
            attack_train,
            attack_calibration,
            "score_only_black_box",
            "logistic_regression",
            ["prob_attack"],
        ),
        build_learned_attack(
            condition,
            attack_train,
            attack_calibration,
            "score_only_black_box",
            "random_forest",
            ["prob_attack"],
        ),
        build_threshold_attack(
            condition,
            attack_calibration,
            "label_aware_audit",
            "loss_threshold",
            "loss",
            -1.0,
        ),
        build_learned_attack(
            condition,
            attack_train,
            attack_calibration,
            "label_aware_audit",
            "logistic_regression",
            ["loss", "correctness"],
        ),
        build_learned_attack(
            condition,
            attack_train,
            attack_calibration,
            "label_aware_audit",
            "random_forest",
            ["loss", "correctness"],
        ),
    ]

    calibration_summary = pd.DataFrame([
        {
            "condition": condition,
            "threat_model": attack.threat_model,
            "attack_model": attack.attack_model,
            "feature_set": attack.feature_set,
            "operating_threshold": attack.operating_threshold,
            "shadow_calibration_auc": attack.shadow_calibration_auc,
            "attacker_training_shadow_seeds": ",".join(
                str(seed)
                for seed in SHADOW_SEEDS
                if seed != CALIBRATION_SHADOW_SEED
            ),
            "calibration_shadow_seed": CALIBRATION_SHADOW_SEED,
        }
        for attack in attacks
    ])

    if SAVE_INTERMEDIATE_ATTACK_DATA:
        attack_data = shadow_data.copy()
        attack_data["attacker_split"] = np.where(
            attack_data["shadow_seed"] == CALIBRATION_SHADOW_SEED,
            "calibration",
            "train",
        )
        attack_data.to_csv(
            INTERMEDIATE_DIR / condition / "shadow_attack_data.csv",
            index=False,
        )

    return attacks, calibration_summary

attacks_by_condition = {}
calibration_frames = []

for condition, shadow_data in shadow_data_by_condition.items():
    condition_attacks, condition_calibration = calibrate_condition_attacks(
        condition,
        shadow_data,
    )
    attacks_by_condition[condition] = condition_attacks
    calibration_frames.append(condition_calibration)

mia_attack_calibration = pd.concat(calibration_frames, ignore_index=True)
mia_attack_calibration.to_csv(
    RESULTS_DIR / "mia_attack_calibration.csv",
    index=False,
)

display(mia_attack_calibration)


,condition,threat_model,attack_model,feature_set,operating_threshold,shadow_calibration_auc,attacker_training_shadow_seeds,calibration_shadow_seed
0,non_private,score_only_black_box,confidence_threshold,confidence,0.999985,0.499482,"101,202,303,404",505
1,non_private,score_only_black_box,logistic_regression,prob_attack,0.499860,0.500934,"101,202,303,404",505
2,non_private,score_only_black_box,random_forest,prob_attack,0.645030,0.498146,"101,202,303,404",505
3,non_private,label_aware_audit,loss_threshold,loss,-0.000015,0.499789,"101,202,303,404",505
4,non_private,label_aware_audit,logistic_regression,loss + correctness,0.501810,0.499777,"101,202,303,404",505
5,non_private,label_aware_audit,random_forest,loss + correctness,0.853904,0.496575,"101,202,303,404",505
6,dp_eps_8,score_only_black_box,confidence_threshold,confidence,0.999996,0.498556,"101,202,303,404",505
7,dp_eps_8,score_only_black_box,logistic_regression,prob_attack,0.499890,0.501358,"101,202,303,404",505
8,dp_eps_8,score_only_black_box,random_forest,prob_attack,0.717649,0.500029,"101,202,303,404",505
9,dp_eps_8,label_aware_audit,loss_threshold,loss,-0.000886,0.498958,"101,202,303,404",505


## 15. Evaluate all fixed attackers on target models


In [23]:
def evaluate_calibrated_attack(
    attack: CalibratedAttack,
    evaluation_data: pd.DataFrame,
    subset_name: str,
    include_confidence_intervals: bool = True,
) -> dict:
    membership_labels = evaluation_data["membership"].to_numpy(dtype=int)
    membership_scores = attack.score_function(evaluation_data)
    membership_predictions = (
        membership_scores >= attack.operating_threshold
    ).astype(int)

    mia_auc = roc_auc_score(membership_labels, membership_scores)
    mia_advantage = calculate_mia_advantage(
        membership_labels,
        membership_scores,
    )

    bootstrap_seed = stable_seed(
        attack.condition,
        attack.threat_model,
        attack.attack_model,
        subset_name,
    )

    if include_confidence_intervals:
        auc_ci = bootstrap_confidence_interval(
            membership_labels,
            membership_scores,
            roc_auc_score,
            bootstrap_seed,
        )
        advantage_ci = bootstrap_confidence_interval(
            membership_labels,
            membership_scores,
            calculate_mia_advantage,
            bootstrap_seed + 1,
        )
    else:
        auc_ci = (np.nan, np.nan)
        advantage_ci = (np.nan, np.nan)

    return {
        "condition": attack.condition,
        "subset": subset_name,
        "threat_model": attack.threat_model,
        "attack_model": attack.attack_model,
        "feature_set": attack.feature_set,
        "n": len(evaluation_data),
        "members": int(membership_labels.sum()),
        "nonmembers": int((1 - membership_labels).sum()),
        "mia_auc": mia_auc,
        "mia_auc_ci_low": auc_ci[0],
        "mia_auc_ci_high": auc_ci[1],
        "mia_advantage": mia_advantage,
        "mia_advantage_ci_low": advantage_ci[0],
        "mia_advantage_ci_high": advantage_ci[1],
        "mia_balanced_accuracy": balanced_accuracy_score(
            membership_labels,
            membership_predictions,
        ),
        "mia_precision": precision_score(
            membership_labels,
            membership_predictions,
            zero_division=0,
        ),
        "mia_recall": recall_score(
            membership_labels,
            membership_predictions,
            zero_division=0,
        ),
        "tpr_at_1pct_fpr": calculate_tpr_at_fpr(
            membership_labels,
            membership_scores,
            0.01,
        ),
        "tpr_at_5pct_fpr": calculate_tpr_at_fpr(
            membership_labels,
            membership_scores,
            0.05,
        ),
        "operating_threshold": attack.operating_threshold,
        "shadow_calibration_auc": attack.shadow_calibration_auc,
    }

overall_mia_rows = []

for condition, attacks in attacks_by_condition.items():
    target_evaluation_data = target_condition_results[
        condition
    ].target_mia_features

    for attack in attacks:
        overall_mia_rows.append(
            evaluate_calibrated_attack(
                attack,
                target_evaluation_data,
                "overall",
                True,
            )
        )

dp_sgd_mia_results = pd.DataFrame(overall_mia_rows).merge(
    dp_sgd_configs[[
        "condition",
        "formal_dp",
        "target_epsilon",
        "actual_epsilon",
        "delta",
        "noise_multiplier",
    ]],
    on="condition",
    how="left",
    validate="many_to_one",
)

dp_sgd_mia_results.to_csv(
    RESULTS_DIR / "dp_sgd_mia_results.csv",
    index=False,
)

display(
    dp_sgd_mia_results.sort_values(
        ["condition", "mia_auc"],
        ascending=[True, False],
    )
)


,condition,subset,threat_model,attack_model,feature_set,n,members,nonmembers,mia_auc,mia_auc_ci_low,mia_auc_ci_high,mia_advantage,mia_advantage_ci_low,mia_advantage_ci_high,mia_balanced_accuracy,mia_precision,mia_recall,tpr_at_1pct_fpr,tpr_at_5pct_fpr,operating_threshold,shadow_calibration_auc,formal_dp,target_epsilon,actual_epsilon,delta,noise_multiplier
19,dp_eps_2,overall,score_only_black_box,logistic_regression,prob_attack,25194,12597,12597,0.502916,0.495996,0.510071,0.011749,0.004590,0.024265,0.500873,0.500864,0.506232,0.000000,0.000000,0.499802,0.501878,True,2.0,1.999038,0.000011,0.881348
21,dp_eps_2,overall,label_aware_audit,loss_threshold,loss,25194,12597,12597,0.501608,0.493912,0.508479,0.011193,0.002934,0.024213,0.503255,0.504265,0.384854,0.000000,0.000000,-0.000017,0.497049,True,2.0,1.999038,0.000011,0.881348
22,dp_eps_2,overall,label_aware_audit,logistic_regression,loss + correctness,25194,12597,12597,0.501608,0.494584,0.508019,0.011193,0.003818,0.023404,0.503255,0.504265,0.384854,0.000000,0.000000,0.500403,0.497049,True,2.0,1.999038,0.000011,0.881348
18,dp_eps_2,overall,score_only_black_box,confidence_threshold,confidence,25194,12597,12597,0.501604,0.494064,0.508711,0.011273,0.003251,0.023440,0.503334,0.504368,0.385012,0.000000,0.000000,0.999983,0.496796,True,2.0,1.999038,0.000011,0.881348
20,dp_eps_2,overall,score_only_black_box,random_forest,prob_attack,25194,12597,12597,0.495136,0.488119,0.501922,0.003493,0.002193,0.012672,0.500278,0.501660,0.083988,0.012146,0.050885,0.712646,0.497306,True,2.0,1.999038,0.000011,0.881348
23,dp_eps_2,overall,label_aware_audit,random_forest,loss + correctness,25194,12597,12597,0.493351,0.486322,0.500240,0.003255,0.002103,0.011003,0.499008,0.497044,0.166865,0.011828,0.050488,0.614969,0.494276,True,2.0,1.999038,0.000011,0.881348
17,dp_eps_4,overall,label_aware_audit,random_forest,loss + correctness,25194,12597,12597,0.503597,0.496905,0.511513,0.013416,0.004949,0.027647,0.500556,0.501211,0.229896,0.008573,0.047075,0.558856,0.502600,True,4.0,3.998267,0.000011,0.681763
14,dp_eps_4,overall,score_only_black_box,random_forest,prob_attack,25194,12597,12597,0.503061,0.496014,0.510202,0.012543,0.005716,0.024732,0.500159,0.500754,0.105422,0.009288,0.050726,0.658049,0.502794,True,4.0,3.998267,0.000011,0.681763
13,dp_eps_4,overall,score_only_black_box,logistic_regression,prob_attack,25194,12597,12597,0.502884,0.496005,0.510136,0.012860,0.004841,0.026136,0.500516,0.500514,0.502262,0.000000,0.000000,0.499856,0.501754,True,4.0,3.998267,0.000011,0.681763
15,dp_eps_4,overall,label_aware_audit,loss_threshold,loss,25194,12597,12597,0.501666,0.493908,0.508831,0.012225,0.003890,0.023758,0.504366,0.505954,0.371041,0.000000,0.000000,-0.000008,0.498117,True,4.0,3.998267,0.000011,0.681763


## 16. Group-conditioned leakage diagnostics


In [24]:
group_analysis_rows = []

for condition, attacks in attacks_by_condition.items():
    target_evaluation_data = target_condition_results[
        condition
    ].target_mia_features

    best_attack_by_threat_model = {}

    for attack in attacks:
        current_best = best_attack_by_threat_model.get(
            attack.threat_model
        )
        if (
            current_best is None
            or attack.shadow_calibration_auc
            > current_best.shadow_calibration_auc
        ):
            best_attack_by_threat_model[
                attack.threat_model
            ] = attack

    for attack in best_attack_by_threat_model.values():
        for group_column in ["binary_group", "family_group"]:
            for group_value, group_data in target_evaluation_data.groupby(
                group_column
            ):
                membership_counts = group_data["membership"].value_counts()

                if (
                    membership_counts.get(0, 0) < MIN_GROUP_NONMEMBERS
                    or membership_counts.get(1, 0) < MIN_GROUP_MEMBERS
                ):
                    continue

                group_analysis_rows.append(
                    evaluate_calibrated_attack(
                        attack,
                        group_data,
                        f"{group_column}={group_value}",
                        True,
                    )
                )

dp_sgd_group_analysis = pd.DataFrame(group_analysis_rows).merge(
    dp_sgd_configs[[
        "condition",
        "formal_dp",
        "target_epsilon",
        "actual_epsilon",
        "delta",
    ]],
    on="condition",
    how="left",
    validate="many_to_one",
)

dp_sgd_group_analysis.to_csv(
    RESULTS_DIR / "dp_sgd_group_analysis.csv",
    index=False,
)

display(dp_sgd_group_analysis)


,condition,subset,threat_model,attack_model,feature_set,n,members,nonmembers,mia_auc,mia_auc_ci_low,mia_auc_ci_high,mia_advantage,mia_advantage_ci_low,mia_advantage_ci_high,mia_balanced_accuracy,mia_precision,mia_recall,tpr_at_1pct_fpr,tpr_at_5pct_fpr,operating_threshold,shadow_calibration_auc,formal_dp,target_epsilon,actual_epsilon,delta
0,non_private,binary_group=Attack,score_only_black_box,logistic_regression,prob_attack,11726,5863,5863,0.504575,0.495308,0.513710,0.012110,0.005693,0.029122,0.500000,0.500000,1.000000,0.000000,0.000000,0.499860,0.500934,False,NaN,NaN,NaN
1,non_private,binary_group=Normal,score_only_black_box,logistic_regression,prob_attack,13468,6734,6734,0.502667,0.493279,0.512051,0.016781,0.006205,0.032400,0.503787,0.508088,0.237897,0.008910,0.049005,0.499860,0.500934,False,NaN,NaN,NaN
2,non_private,family_group=DoS,score_only_black_box,logistic_regression,prob_attack,9186,4593,4593,0.502974,0.494813,0.511625,0.007620,0.001036,0.023447,0.500000,0.500000,1.000000,0.000000,0.000000,0.499860,0.500934,False,NaN,NaN,NaN
3,non_private,family_group=Normal,score_only_black_box,logistic_regression,prob_attack,13468,6734,6734,0.502667,0.492457,0.512592,0.016781,0.007528,0.032285,0.503787,0.508088,0.237897,0.008910,0.049005,0.499860,0.500934,False,NaN,NaN,NaN
4,non_private,family_group=Probe,score_only_black_box,logistic_regression,prob_attack,2332,1166,1166,0.516704,0.494736,0.540191,0.036021,0.020231,0.079941,0.500000,0.500000,1.000000,0.000000,0.000000,0.499860,0.500934,False,NaN,NaN,NaN
5,non_private,family_group=Rare,score_only_black_box,logistic_regression,prob_attack,208,104,104,0.646219,0.567568,0.722829,0.240385,0.163413,0.386262,0.500000,0.500000,1.000000,0.086538,0.182692,0.499860,0.500934,False,NaN,NaN,NaN
6,non_private,binary_group=Attack,label_aware_audit,loss_threshold,loss,11726,5863,5863,0.504575,0.495700,0.513512,0.012110,0.006107,0.029030,0.500085,0.500051,0.833873,0.000000,0.000000,-0.000015,0.499789,False,NaN,NaN,NaN
7,non_private,binary_group=Normal,label_aware_audit,loss_threshold,loss,13468,6734,6734,0.497376,0.487918,0.507229,0.005792,0.002279,0.024676,0.500000,0.500000,0.433175,0.000000,0.032670,-0.000015,0.499789,False,NaN,NaN,NaN
8,non_private,family_group=DoS,label_aware_audit,loss_threshold,loss,9186,4593,4593,0.502974,0.494632,0.511849,0.007620,0.001301,0.022947,0.499673,0.499827,0.944481,0.000000,0.000000,-0.000015,0.499789,False,NaN,NaN,NaN
9,non_private,family_group=Normal,label_aware_audit,loss_threshold,loss,13468,6734,6734,0.497376,0.487115,0.507209,0.005792,0.002298,0.025944,0.500000,0.500000,0.433175,0.000000,0.032670,-0.000015,0.499789,False,NaN,NaN,NaN


## 17. Paired DP-minus-non-private MIA differences

Negative differences indicate lower measured leakage under DP. A reduction is supported only when the complete paired 95% confidence interval is below zero.


In [25]:
def best_attack_by_threat(attacks):
    selected = {}
    for attack in attacks:
        current = selected.get(attack.threat_model)
        if current is None or attack.shadow_calibration_auc > current.shadow_calibration_auc:
            selected[attack.threat_model] = attack
    return selected

keys = ["membership", "partition_position", "row_id"]
reference_data = (
    target_condition_results["non_private"].target_mia_features
    .sort_values(keys)
    .reset_index(drop=True)
)
reference_attacks = best_attack_by_threat(attacks_by_condition["non_private"])
paired_rows = []

for dp_condition in ["dp_eps_8", "dp_eps_4", "dp_eps_2"]:
    comparison_data = (
        target_condition_results[dp_condition].target_mia_features
        .sort_values(keys)
        .reset_index(drop=True)
    )
    pd.testing.assert_frame_equal(
        reference_data[keys],
        comparison_data[keys],
        check_dtype=False,
    )
    comparison_attacks = best_attack_by_threat(attacks_by_condition[dp_condition])

    masks = {"overall": np.ones(len(reference_data), dtype=bool)}
    for column in ["binary_group", "family_group"]:
        for value in sorted(reference_data[column].unique()):
            masks[f"{column}={value}"] = (
                reference_data[column].to_numpy() == value
            )

    for threat_model in ["score_only_black_box", "label_aware_audit"]:
        ref_attack = reference_attacks[threat_model]
        dp_attack = comparison_attacks[threat_model]
        ref_scores_all = ref_attack.score_function(reference_data)
        dp_scores_all = dp_attack.score_function(comparison_data)

        for subset, mask in masks.items():
            labels = reference_data.loc[mask, "membership"].to_numpy(dtype=int)
            counts = pd.Series(labels).value_counts()
            if counts.get(0, 0) < MIN_GROUP_NONMEMBERS or counts.get(1, 0) < MIN_GROUP_MEMBERS:
                continue

            for metric_name, metric_function in {
                "mia_auc": roc_auc_score,
                "mia_advantage": calculate_mia_advantage,
            }.items():
                difference, ci_low, ci_high = paired_bootstrap_metric_difference(
                    labels,
                    ref_scores_all[mask],
                    dp_scores_all[mask],
                    metric_function,
                    stable_seed("paired", dp_condition, threat_model, subset, metric_name),
                )
                paired_rows.append({
                    "reference_condition": "non_private",
                    "comparison_condition": dp_condition,
                    "threat_model": threat_model,
                    "subset": subset,
                    "metric": metric_name,
                    "n": int(mask.sum()),
                    "reference_attack_model": ref_attack.attack_model,
                    "comparison_attack_model": dp_attack.attack_model,
                    "reference_estimate": metric_function(labels, ref_scores_all[mask]),
                    "comparison_estimate": metric_function(labels, dp_scores_all[mask]),
                    "difference_dp_minus_non_private": difference,
                    "difference_ci_low": ci_low,
                    "difference_ci_high": ci_high,
                    "supports_measured_reduction": bool(ci_high < 0),
                    "bootstrap_n": BOOTSTRAP_N,
                })

dp_sgd_paired_mia_differences = pd.DataFrame(paired_rows).merge(
    dp_sgd_configs[[
        "condition",
        "target_epsilon",
        "actual_epsilon",
        "delta",
    ]],
    left_on="comparison_condition",
    right_on="condition",
    how="left",
    validate="many_to_one",
).drop(columns=["condition"])

dp_sgd_paired_mia_differences.to_csv(
    RESULTS_DIR / "dp_sgd_paired_mia_differences.csv",
    index=False,
)

display(dp_sgd_paired_mia_differences[
    (dp_sgd_paired_mia_differences["metric"] == "mia_auc")
    & dp_sgd_paired_mia_differences["subset"].isin(
        ["overall", "family_group=Rare"]
    )
])


,reference_condition,comparison_condition,threat_model,subset,metric,n,reference_attack_model,comparison_attack_model,reference_estimate,comparison_estimate,difference_dp_minus_non_private,difference_ci_low,difference_ci_high,supports_measured_reduction,bootstrap_n,target_epsilon,actual_epsilon,delta
0,non_private,dp_eps_8,score_only_black_box,overall,mia_auc,25194,logistic_regression,logistic_regression,0.501777,0.502819,0.001042,-0.001581,0.003820,False,1000,8.0,7.993649,0.000011
12,non_private,dp_eps_8,score_only_black_box,family_group=Rare,mia_auc,208,logistic_regression,logistic_regression,0.646219,0.567123,-0.079096,-0.137144,-0.024662,True,1000,8.0,7.993649,0.000011
14,non_private,dp_eps_8,label_aware_audit,overall,mia_auc,25194,loss_threshold,loss_threshold,0.501439,0.502031,0.000593,-0.003346,0.004763,False,1000,8.0,7.993649,0.000011
26,non_private,dp_eps_8,label_aware_audit,family_group=Rare,mia_auc,208,loss_threshold,loss_threshold,0.646219,0.567123,-0.079096,-0.134630,-0.022655,True,1000,8.0,7.993649,0.000011
28,non_private,dp_eps_4,score_only_black_box,overall,mia_auc,25194,logistic_regression,random_forest,0.501777,0.503061,0.001284,-0.009119,0.011828,False,1000,4.0,3.998267,0.000011
40,non_private,dp_eps_4,score_only_black_box,family_group=Rare,mia_auc,208,logistic_regression,random_forest,0.646219,0.523530,-0.122689,-0.230004,-0.004819,True,1000,4.0,3.998267,0.000011
42,non_private,dp_eps_4,label_aware_audit,overall,mia_auc,25194,loss_threshold,random_forest,0.501439,0.503597,0.002158,-0.007619,0.012015,False,1000,4.0,3.998267,0.000011
54,non_private,dp_eps_4,label_aware_audit,family_group=Rare,mia_auc,208,loss_threshold,random_forest,0.646219,0.547152,-0.099066,-0.209033,0.017027,False,1000,4.0,3.998267,0.000011
56,non_private,dp_eps_2,score_only_black_box,overall,mia_auc,25194,logistic_regression,logistic_regression,0.501777,0.502916,0.001139,-0.001800,0.004203,False,1000,2.0,1.999038,0.000011
68,non_private,dp_eps_2,score_only_black_box,family_group=Rare,mia_auc,208,logistic_regression,logistic_regression,0.646219,0.564626,-0.081592,-0.145588,-0.025276,True,1000,2.0,1.999038,0.000011


## 18. Bootstrap CI table and privacy–utility summary


In [26]:
bootstrap_rows = []

for _, row in dp_sgd_mia_results.iterrows():
    for metric_name in ["mia_auc", "mia_advantage"]:
        bootstrap_rows.append({
            "condition": row["condition"],
            "formal_dp": row["formal_dp"],
            "actual_epsilon": row["actual_epsilon"],
            "delta": row["delta"],
            "threat_model": row["threat_model"],
            "attack_model": row["attack_model"],
            "metric": metric_name,
            "estimate": row[metric_name],
            "ci_low": row[f"{metric_name}_ci_low"],
            "ci_high": row[f"{metric_name}_ci_high"],
            "bootstrap_n": BOOTSTRAP_N,
        })

dp_sgd_bootstrap_ci = pd.DataFrame(bootstrap_rows)
dp_sgd_bootstrap_ci.to_csv(
    RESULTS_DIR / "dp_sgd_bootstrap_ci.csv",
    index=False,
)

display(dp_sgd_bootstrap_ci.head())


,condition,formal_dp,actual_epsilon,delta,threat_model,attack_model,metric,estimate,ci_low,ci_high,bootstrap_n
0,non_private,False,NaN,NaN,score_only_black_box,confidence_threshold,mia_auc,0.501396,0.494364,0.508441,1000
1,non_private,False,NaN,NaN,score_only_black_box,confidence_threshold,mia_advantage,0.005636,0.002683,0.018942,1000
2,non_private,False,NaN,NaN,score_only_black_box,logistic_regression,mia_auc,0.501777,0.494926,0.508914,1000
3,non_private,False,NaN,NaN,score_only_black_box,logistic_regression,mia_advantage,0.008970,0.002963,0.019749,1000
4,non_private,False,NaN,NaN,score_only_black_box,random_forest,mia_auc,0.495144,0.488250,0.501545,1000


In [27]:
best_mia_rows = (
    dp_sgd_mia_results
    .sort_values(
        ["condition", "threat_model", "shadow_calibration_auc"],
        ascending=[True, True, False],
    )
    .groupby(["condition", "threat_model"], as_index=False)
    .head(1)
)

tuned_ids_rows = dp_sgd_ids_results[
    (dp_sgd_ids_results["split"] == "KDDTest+")
    & (
        dp_sgd_ids_results["threshold_policy"]
        == "validation_selected_F2"
    )
].copy()

privacy_utility_summary = best_mia_rows.merge(
    tuned_ids_rows,
    on=[
        "condition",
        "formal_dp",
        "actual_epsilon",
        "delta",
    ],
    how="left",
    validate="many_to_one",
    suffixes=("_mia", "_ids"),
)

privacy_utility_summary.to_csv(
    RESULTS_DIR / "dp_sgd_privacy_utility_summary.csv",
    index=False,
)

display(privacy_utility_summary)


,condition,subset,threat_model,attack_model,feature_set,n,members,nonmembers,mia_auc,mia_auc_ci_low,mia_auc_ci_high,mia_advantage,mia_advantage_ci_low,mia_advantage_ci_high,mia_balanced_accuracy,mia_precision,mia_recall,tpr_at_1pct_fpr,tpr_at_5pct_fpr,operating_threshold,shadow_calibration_auc,formal_dp,target_epsilon,actual_epsilon,delta,noise_multiplier,split,threshold_policy,threshold,accuracy,precision,recall,f1,fnr,fpr,roc_auc,pr_auc,tn,fp,fn,tp
0,dp_eps_2,overall,label_aware_audit,loss_threshold,loss,25194,12597,12597,0.501608,0.493912,0.508479,0.011193,0.002934,0.024213,0.503255,0.504265,0.384854,0.000000,0.000000,-0.000017,0.497049,True,2.0,1.999038,0.000011,0.881348,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,dp_eps_2,overall,score_only_black_box,logistic_regression,prob_attack,25194,12597,12597,0.502916,0.495996,0.510071,0.011749,0.004590,0.024265,0.500873,0.500864,0.506232,0.000000,0.000000,0.499802,0.501878,True,2.0,1.999038,0.000011,0.881348,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,dp_eps_4,overall,label_aware_audit,random_forest,loss + correctness,25194,12597,12597,0.503597,0.496905,0.511513,0.013416,0.004949,0.027647,0.500556,0.501211,0.229896,0.008573,0.047075,0.558856,0.502600,True,4.0,3.998267,0.000011,0.681763,KDDTest+,validation_selected_F2,0.02,0.806645,0.916290,0.726720,0.810569,0.273280,0.087736,0.838832,0.897761,8859.0,852.0,3507.0,9326.0
3,dp_eps_4,overall,score_only_black_box,random_forest,prob_attack,25194,12597,12597,0.503061,0.496014,0.510202,0.012543,0.005716,0.024732,0.500159,0.500754,0.105422,0.009288,0.050726,0.658049,0.502794,True,4.0,3.998267,0.000011,0.681763,KDDTest+,validation_selected_F2,0.02,0.806645,0.916290,0.726720,0.810569,0.273280,0.087736,0.838832,0.897761,8859.0,852.0,3507.0,9326.0
4,dp_eps_8,overall,label_aware_audit,loss_threshold,loss,25194,12597,12597,0.502031,0.494922,0.508584,0.011114,0.003187,0.023426,0.500595,0.500394,0.755894,0.000000,0.000000,-0.000886,0.498958,True,8.0,7.993649,0.000011,0.560303,KDDTest+,validation_selected_F2,0.03,0.801278,0.920129,0.712772,0.803284,0.287228,0.081763,0.837541,0.896710,8917.0,794.0,3686.0,9147.0
5,dp_eps_8,overall,score_only_black_box,logistic_regression,prob_attack,25194,12597,12597,0.502819,0.496049,0.509291,0.012146,0.004591,0.024208,0.500079,0.500080,0.497023,0.000000,0.000000,0.499890,0.501358,True,8.0,7.993649,0.000011,0.560303,KDDTest+,validation_selected_F2,0.03,0.801278,0.920129,0.712772,0.803284,0.287228,0.081763,0.837541,0.896710,8917.0,794.0,3686.0,9147.0
6,non_private,overall,label_aware_audit,loss_threshold,loss,25194,12597,12597,0.501439,0.494269,0.508487,0.005636,0.003476,0.018977,0.500040,0.500032,0.619671,0.000000,0.000000,-0.000015,0.499789,False,NaN,NaN,NaN,NaN,KDDTest+,validation_selected_F2,0.29,0.816537,0.958945,0.708018,0.814596,0.291982,0.040058,0.910364,0.935714,9322.0,389.0,3747.0,9086.0
7,non_private,overall,score_only_black_box,logistic_regression,prob_attack,25194,12597,12597,0.501777,0.494926,0.508914,0.008970,0.002963,0.019749,0.502024,0.501714,0.592601,0.000000,0.000000,0.499860,0.500934,False,NaN,NaN,NaN,NaN,KDDTest+,validation_selected_F2,0.29,0.816537,0.958945,0.708018,0.814596,0.291982,0.040058,0.910364,0.935714,9322.0,389.0,3747.0,9086.0


## 19. Save deduplicated warnings, config, and manifest


In [28]:
for result in target_condition_results.values():
    all_warning_rows.extend(result.warning_rows)

warning_table = pd.DataFrame(
    all_warning_rows,
    columns=[
        "stage",
        "condition",
        "shadow_seed",
        "category",
        "message",
        "occurrences",
    ],
)

if not warning_table.empty:
    warning_table = (
        warning_table.groupby(
            [
                "stage",
                "condition",
                "shadow_seed",
                "category",
                "message",
            ],
            dropna=False,
            as_index=False,
        )["occurrences"]
        .sum()
    )

warning_table.to_csv(
    RESULTS_DIR / "opacus_warning_summary.csv",
    index=False,
)

experiment_config = {
    "protocol_version": "ROADMAP_REVISED_V2",
    "protocol_revision": SHADOW_PROTOCOL_VERSION,
    "experiment": "05_dp_sgd_privacy_utility_sweep",
    "seed": SEED,
    "shadow_seeds": SHADOW_SEEDS,
    "calibration_shadow_seed": CALIBRATION_SHADOW_SEED,
    "conditions": CONDITIONS,
    "target_epsilons": TARGET_EPSILONS,
    "target_delta": TARGET_DELTA,
    "architecture": {
        "target_input_dim": TARGET_INPUT_DIM,
        "hidden_dims": list(HIDDEN_DIMS),
        "activation": "ReLU",
        "output": "one logit",
        "batch_norm": False,
    },
    "training": {
        "optimizer": "Adam",
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "loss": "BCEWithLogitsLoss",
    },
    "dp": {
        "max_grad_norm": MAX_GRAD_NORM,
        "accountant": ACCOUNTANT,
        "poisson_sampling": POISSON_SAMPLING,
        "secure_mode": SECURE_MODE,
        "privacy_scope": (
            "DP-SGD optimisation conditional on fixed preprocessing."
        ),
    },
    "mia": {
        "bootstrap_n": BOOTSTRAP_N,
        "threat_models": [
            "score_only_black_box",
            "label_aware_audit",
        ],
        "attacks": [
            "threshold",
            "logistic_regression",
            "random_forest",
        ],
        "target_evaluation": (
            "Balanced target_train members and "
            "target_validation non-members."
        ),
        "shadow_protocol": (
            "Five condition-matched shadows; seeds 101-404 "
            "train attackers and seed 505 calibrates thresholds."
        ),
        "dp_shadow_rule": (
            "Match target epsilon and fixed target delta; "
            "derive a shadow-specific noise multiplier."
        ),
        "paired_difference_rule": (
            "Paired bootstrap on identical target records."
        ),
    },
    "execution": {
        "resume": RESUME,
        "save_intermediate_attack_data": SAVE_INTERMEDIATE_ATTACK_DATA,
    },
}

write_strict_json(RESULTS_DIR / "config.json", experiment_config)


In [29]:
required_output_files = [
    RESULTS_DIR / "dp_sgd_ids_results.csv",
    RESULTS_DIR / "dp_sgd_mia_results.csv",
    RESULTS_DIR / "dp_sgd_privacy_utility_summary.csv",
    RESULTS_DIR / "dp_sgd_configs.csv",
    RESULTS_DIR / "dp_sgd_bootstrap_ci.csv",
    RESULTS_DIR / "dp_sgd_group_analysis.csv",
    RESULTS_DIR / "dp_sgd_paired_mia_differences.csv",
    RESULTS_DIR / "mia_attack_calibration.csv",
    RESULTS_DIR / "shadow_model_configs.csv",
    RESULTS_DIR / "target_mia_sample_manifest.csv",
    RESULTS_DIR / "opacus_warning_summary.csv",
    RESULTS_DIR / "config.json",
]

manifest = {
    **experiment_config,
    "created_at_unix": time.time(),
    "device": str(DEVICE),
    "dataset": {
        "train_file": str(TRAIN_FILE),
        "test_file": str(TEST_FILE),
        "train_sha256": actual_train_hash,
        "test_sha256": actual_test_hash,
    },
    "split": {
        "target_train_rows": len(target_train),
        "target_validation_rows": len(target_validation),
        "shadow_pool_rows": len(shadow_pool),
    },
    "preprocessor": {
        "path": str(PREPROCESSOR_FILE),
        "sha256": preprocessor_hash,
    },
    "prerequisites": {
        "baseline_manifest": str(BASELINE_MANIFEST_FILE),
        "smoke_manifest": str(SMOKE_MANIFEST_FILE),
    },
    "target_condition_configs": dp_sgd_configs.to_dict(
        orient="records"
    ),
    "software": {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "sklearn": sklearn.__version__,
        "torch": torch.__version__,
        "opacus": __import__("opacus").__version__,
    },
    "required_outputs": [str(path) for path in required_output_files],
}

manifest_path = RESULTS_DIR / "dp_sgd_sweep_manifest.json"

write_strict_json(manifest_path, manifest)
write_strict_json(
    MANIFEST_DIR / "dp_sgd_sweep_manifest.json",
    manifest,
)

print("Manifest saved:", manifest_path)


Manifest saved: /content/drive/MyDrive/ML-DP-NID/results/dp_sgd/dp_sgd_sweep_manifest.json


## 20. Final protocol and output gate


In [30]:
missing_outputs = [
    str(path)
    for path in required_output_files
    if not path.exists()
]

expected_conditions = {
    "non_private",
    "dp_eps_8",
    "dp_eps_4",
    "dp_eps_2",
}
completed_conditions = set(dp_sgd_configs["condition"])
condition_gate = completed_conditions == expected_conditions

epsilon_gate = True
for _, config_row in dp_sgd_configs[
    dp_sgd_configs["formal_dp"]
].iterrows():
    target_epsilon = float(config_row["target_epsilon"])
    actual_epsilon = float(config_row["actual_epsilon"])
    epsilon_gate = (
        epsilon_gate
        and np.isfinite(actual_epsilon)
        and actual_epsilon > 0
        and actual_epsilon <= target_epsilon + 0.10
    )

expected_overall_mia_rows = len(expected_conditions) * 6
mia_gate = len(dp_sgd_mia_results) == expected_overall_mia_rows

paired_gate = bool(
    not dp_sgd_paired_mia_differences.empty
    and set(dp_sgd_paired_mia_differences["comparison_condition"])
    == {"dp_eps_8", "dp_eps_4", "dp_eps_2"}
)

final_gate = bool(
    condition_gate
    and epsilon_gate
    and shadow_budget_gate
    and mia_gate
    and paired_gate
    and not missing_outputs
)

print({
    "completed_conditions": sorted(completed_conditions),
    "condition_gate": condition_gate,
    "epsilon_gate": epsilon_gate,
    "overall_mia_rows": len(dp_sgd_mia_results),
    "expected_overall_mia_rows": expected_overall_mia_rows,
    "shadow_budget_gate": shadow_budget_gate,
    "mia_gate": mia_gate,
    "paired_difference_gate": paired_gate,
    "all_outputs_saved": not missing_outputs,
    "experiment_05_protocol_gate": final_gate,
})

if not final_gate:
    raise RuntimeError(
        "Experiment 05 protocol gate failed. "
        "Do not interpret or select a DP setting."
    )

print(
    "\nExperiment 05 completed. "
    "Do not add ε≈1 or start repeated runs "
    "until these outputs are reviewed."
)


{'completed_conditions': ['dp_eps_2', 'dp_eps_4', 'dp_eps_8', 'non_private'], 'condition_gate': True, 'epsilon_gate': True, 'overall_mia_rows': 24, 'expected_overall_mia_rows': 24, 'shadow_budget_gate': True, 'mia_gate': True, 'paired_difference_gate': True, 'all_outputs_saved': True, 'experiment_05_protocol_gate': True}

Experiment 05 completed. Do not add ε≈1 or start repeated runs until these outputs are reviewed.


## 21. Interpretation boundary

- Reused target models mean target IDS results should remain unchanged.
- Every DP shadow must match its requested epsilon within 0.10 and use the fixed target delta.
- Claim a measured MIA reduction only when the paired confidence interval is fully below zero.
- Keep Rare-group findings exploratory because the subgroup is small and multiple comparisons are made.
- Do not add epsilon approximately 1 before the corrected results are reviewed.
